# Test 1: Proper 3-Way Train / Val / Test Split

**Goal**: Evaluate the GraphCodeBERT model on a **held-out test set** never seen during training.

The original notebook uses a 90/10 train/val split and reports validation accuracy as the final score.
This is problematic because the best checkpoint is selected based on validation performance,
making the validation score optimistic. A clean test set gives unbiased final metrics.

**Split**: 72% train · 8% val (for checkpoint selection) · 20% test (final report only)

**Expected Kaggle inputs**:
- `/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl`
- `/kaggle/input/model2/model.bin` (pre-trained GraphCodeBERT weights)

**Output**: Final test-set metrics saved to `/kaggle/working/test1_results.txt`

In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 16.0 MB/s eta 0:00:00


In [3]:
import os
import json
import torch
import logging
import random
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from transformers import (
    get_linear_schedule_with_warmup,
    RobertaConfig, RobertaModel,
    AutoTokenizer)

from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    classification_report, confusion_matrix
)

print("All imports successful.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

All imports successful.
PyTorch version: 2.9.0+cu126
CUDA available: True


In [4]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
class Args:
    output_dir          = "saved_models_test1"
    model_name_or_path  = "microsoft/graphcodebert-base"
    tokenizer_name      = "microsoft/graphcodebert-base"
    train_file          = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
    pretrained_weights  = "/kaggle/input/datasets/hasanmahmudabdullah/model2/model.bin"  # Optional: skip training

    # Sequence lengths
    code_length         = 384
    data_flow_length    = 128

    # Training
    train_batch_size    = 16
    eval_batch_size     = 32
    learning_rate       = 2e-5
    max_grad_norm       = 1.0
    num_train_epochs    = 3
    seed                = 42

    # ── KEY DIFFERENCE from baseline notebook ──
    # 3-way split: 72% train, 8% val, 20% test
    train_ratio         = 0.72
    val_ratio           = 0.08
    # test_ratio = 1.0 - train_ratio - val_ratio = 0.20

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu  = torch.cuda.device_count()

args = Args()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"Device: {args.device}  |  GPUs: {args.n_gpu}")
print(f"Split: {args.train_ratio*100:.0f}% train / "
      f"{args.val_ratio*100:.0f}% val / "
      f"{(1-args.train_ratio-args.val_ratio)*100:.0f}% test")

Device: cuda  |  GPUs: 1
Split: 72% train / 8% val / 20% test


In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if args.n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

set_seed(args.seed)

In [6]:
# ─── MODEL DEFINITION (identical to graphcodebert-training.ipynb) ─────────────
class Model(nn.Module):
    def __init__(self, encoder, config, tokenizer, args):
        super(Model, self).__init__()
        self.encoder   = encoder
        self.config    = config
        self.tokenizer = tokenizer
        self.args      = args
        self.dropout   = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        extended_attention_mask = (1.0 - attn_mask) * -10000.0
        extended_attention_mask = extended_attention_mask.unsqueeze(1)

        embedding_output = self.encoder.embeddings(
            input_ids=input_ids,
            position_ids=p_ids
        )
        encoder_outputs = self.encoder.encoder(
            embedding_output,
            attention_mask=extended_attention_mask,
            head_mask=[None] * self.config.num_hidden_layers
        )
        sequence_output = encoder_outputs[0]
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)

        if labels is not None:
            loss = CrossEntropyLoss()(logits, labels)
            return loss, prob
        return prob

In [7]:
# ─── DATASET (identical to graphcodebert-training.ipynb) ──────────────────────
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args      = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        with open(file_path, 'r') as f:
            self.lines = f.readlines()

    def __len__(self):
        return len(self.lines)

    def _get_char_index(self, code_lines, coord):
        row, col = coord
        char_idx = 0
        for i in range(min(row, len(code_lines))):
            char_idx += len(code_lines[i])
        return char_idx + col

    def __getitem__(self, item):
        line  = self.lines[item]
        entry = json.loads(line)
        code  = entry.get('code', '')
        dfg   = entry.get('dfg', [])[:self.args.data_flow_length]
        label = int(entry.get('label', 0)) if entry.get('label') is not None else 0

        tokens_obj = self.tokenizer(
            code,
            max_length=self.args.code_length,
            truncation=True,
            padding='max_length',
            return_offsets_mapping=True
        )
        input_ids  = tokens_obj['input_ids']
        offsets    = tokens_obj['offset_mapping']
        code_lines = code.splitlines(keepends=True)

        dfg_ids         = [self.tokenizer.unk_token_id] * len(dfg)
        node_to_token_map = {}
        pos_to_node_idx   = {}

        for node_idx, node in enumerate(dfg):
            start_pos = node[1][0]
            end_pos   = node[1][1]
            pos_key   = (start_pos[0], start_pos[1], end_pos[0], end_pos[1])
            pos_to_node_idx[pos_key] = node_idx
            try:
                char_start = self._get_char_index(code_lines, start_pos)
                char_end   = self._get_char_index(code_lines, end_pos)
                aligned    = []
                for t_idx, (t_start, t_end) in enumerate(offsets):
                    if t_start == t_end:
                        continue
                    if (t_start >= char_start and t_end <= char_end) or \
                       (char_start >= t_start and char_end <= t_end):
                        aligned.append(t_idx)
                node_to_token_map[node_idx] = aligned
            except IndexError:
                node_to_token_map[node_idx] = []

        c_len    = self.args.code_length
        attn_mask = np.zeros((self.total_len, self.total_len), dtype=bool)
        attn_mask[:c_len, :c_len] = True

        for node_idx, node in enumerate(dfg):
            abs_node_idx = c_len + node_idx
            for t_idx in node_to_token_map.get(node_idx, []):
                attn_mask[abs_node_idx, t_idx] = True
                attn_mask[t_idx, abs_node_idx] = True
            for p_pos in node[4]:
                p_key = (p_pos[0][0], p_pos[0][1], p_pos[1][0], p_pos[1][1])
                if p_key in pos_to_node_idx:
                    abs_parent = c_len + pos_to_node_idx[p_key]
                    attn_mask[abs_node_idx, abs_parent] = True
                    attn_mask[abs_parent, abs_node_idx] = True
            attn_mask[abs_node_idx, abs_node_idx] = True

        full_input_ids = input_ids + dfg_ids
        p_ids          = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)
        pad_len        = self.total_len - len(full_input_ids)
        if pad_len > 0:
            full_input_ids += [self.tokenizer.pad_token_id] * pad_len
            p_ids          += [1] * pad_len

        return {
            'input_ids': torch.tensor(full_input_ids, dtype=torch.long),
            'p_ids':     torch.tensor(p_ids,          dtype=torch.long),
            'attn_mask': torch.tensor(attn_mask,      dtype=torch.float),
            'label':     torch.tensor(label,           dtype=torch.long)
        }

In [8]:
# ─── 3-WAY SPLIT ─────────────────────────────────────────────────────────────
def create_splits(dataset, args):
    """Reproducible 3-way split: train / val / test."""
    n         = len(dataset)
    train_n   = int(args.train_ratio * n)
    val_n     = int(args.val_ratio   * n)
    test_n    = n - train_n - val_n

    generator = torch.Generator().manual_seed(args.seed)
    train_ds, val_ds, test_ds = random_split(
        dataset, [train_n, val_n, test_n], generator=generator
    )

    print(f"Dataset sizes — train: {len(train_ds):,}  "
          f"val: {len(val_ds):,}  "
          f"test: {len(test_ds):,}")
    return train_ds, val_ds, test_ds

In [9]:
# ─── EVALUATION HELPER ───────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, dataset, desc="Evaluating"):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size,
                        num_workers=2, pin_memory=True)
    model.eval()
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc=desc):
        inputs = {
            'input_ids': batch['input_ids'].to(args.device),
            'p_ids':     batch['p_ids'].to(args.device),
            'attn_mask': batch['attn_mask'].to(args.device)
        }
        probs = model(**inputs)
        all_preds.extend(torch.argmax(probs, dim=-1).cpu().numpy())
        all_labels.extend(batch['label'].numpy())

    model.train()
    return accuracy_score(all_labels, all_preds), all_preds, all_labels

In [10]:
# ─── TRAINING LOOP ───────────────────────────────────────────────────────────
def train(model, train_ds, val_ds):
    loader = DataLoader(
        train_ds, batch_size=args.train_batch_size,
        shuffle=True, num_workers=2, pin_memory=True
    )
    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=len(loader) * args.num_train_epochs
    )
    scaler   = GradScaler()
    best_acc = 0.0

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0.0
        for batch in tqdm(loader, desc=f"Epoch {epoch}"):
            inputs = {
                'input_ids': batch['input_ids'].to(args.device),
                'p_ids':     batch['p_ids'].to(args.device),
                'attn_mask': batch['attn_mask'].to(args.device),
                'labels':    batch['label'].to(args.device)
            }
            optimizer.zero_grad()
            with autocast():
                loss, _ = model(**inputs)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss += loss.item()

        val_acc, _, _ = evaluate(model, val_ds, desc=f"  Val epoch {epoch}")
        avg_loss      = tr_loss / len(loader)
        logger.info(f"Epoch {epoch} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4%}")

        if val_acc > best_acc:
            best_acc = val_acc
            os.makedirs(args.output_dir, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(args.output_dir, "model.bin"))
            logger.info(f"  ✓ New best val acc: {best_acc:.4%} — model saved.")

    logger.info(f"Training complete. Best val acc: {best_acc:.4%}")
    return best_acc

In [11]:
# ─── MAIN ─────────────────────────────────────────────────────────────────────
set_seed(args.seed)

# 1. Load model
config    = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name)
encoder   = RobertaModel.from_pretrained(args.model_name_or_path, config=config)

model = Model(encoder, config, tokenizer, args).to(args.device)

# 2. Dataset & 3-way split
full_ds                  = TextDataset(tokenizer, args, args.train_file)
train_ds, val_ds, test_ds = create_splits(full_ds, args)

# 3. OPTION A: Load pre-trained weights and skip training (fast path)
USE_PRETRAINED = os.path.exists(args.pretrained_weights)

if USE_PRETRAINED:
    print(f"Loading pre-trained weights from {args.pretrained_weights}")
    model.load_state_dict(torch.load(args.pretrained_weights, map_location=args.device))
    print("Weights loaded. Skipping training — evaluating directly on test set.")
else:
    # 3. OPTION B: Train from scratch
    print("No pre-trained weights found. Training from scratch...")
    best_val_acc = train(model, train_ds, val_ds)

    # Reload best checkpoint
    best_ckpt = os.path.join(args.output_dir, "model.bin")
    if os.path.exists(best_ckpt):
        print(f"Reloading best checkpoint from {best_ckpt}")
        model.load_state_dict(torch.load(best_ckpt, map_location=args.device))

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/vocab.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/graphcodebert-base/2b0488a7bb0eefc7041f1bb2cad1ab26b0da269d/vocab.json "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/graphcodebert-base/commits/refs%2Fpr%2F8 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/graphcodebert-base/resolve/refs%2Fpr%2F8/model.safetensors.index.json "HTTP/1.1 404 Not Found"
RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Dataset sizes — train: 143,971  val: 15,996  test: 39,993
Loading pre-trained weights from /kaggle/input/datasets/hasanmahmudabdullah/model2/model.bin
Weights loaded. Skipping training — evaluating directly on test set.


In [12]:
# ─── FINAL TEST SET EVALUATION ───────────────────────────────────────────────
print("\n" + "="*50)
print("FINAL TEST SET RESULTS (held-out, never seen during training)")
print("="*50)

test_acc, test_preds, test_labels = evaluate(model, test_ds, desc="Test set")

print(f"\nTest Accuracy: {test_acc:.4%}")
print("-" * 50)
print(classification_report(
    test_labels, test_preds,
    target_names=['Safe (0)', 'Malicious (1)'],
    digits=4
))

cm = confusion_matrix(test_labels, test_preds)
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:")
print(f"  True  Negatives  (Safe correctly identified): {tn}")
print(f"  False Positives  (Safe flagged as Malicious):  {fp}")
print(f"  False Negatives  (Malicious missed):           {fn}")
print(f"  True  Positives  (Malicious detected):        {tp}")

# ─── COMPARE: val-set score vs test-set score ────────────────────────────────
print("\n" + "="*50)
print("OPTIMISM BIAS CHECK")
print("="*50)
val_acc, _, _ = evaluate(model, val_ds, desc="Val set")
print(f"Val  Accuracy (used for checkpoint selection): {val_acc:.4%}")
print(f"Test Accuracy (true held-out estimate):        {test_acc:.4%}")
bias = val_acc - test_acc
print(f"Optimism Bias (val - test):                    {bias:+.4%}")
if bias > 0.01:
    print("⚠  Val acc overestimates true performance by > 1%. Use test acc in papers.")
else:
    print("✓  Val and test acc are close — data split looks healthy.")


FINAL TEST SET RESULTS (held-out, never seen during training)


Test set: 100%|██████████| 1250/1250 [10:22<00:00,  2.01it/s]



Test Accuracy: 92.0161%
--------------------------------------------------
               precision    recall  f1-score   support

     Safe (0)     0.9191    0.9232    0.9211     20202
Malicious (1)     0.9212    0.9171    0.9192     19791

     accuracy                         0.9202     39993
    macro avg     0.9202    0.9201    0.9201     39993
 weighted avg     0.9202    0.9202    0.9202     39993

Confusion Matrix:
  True  Negatives  (Safe correctly identified): 18650
  False Positives  (Safe flagged as Malicious):  1552
  False Negatives  (Malicious missed):           1641
  True  Positives  (Malicious detected):        18150

OPTIMISM BIAS CHECK


Val set: 100%|██████████| 500/500 [04:09<00:00,  2.01it/s]

Val  Accuracy (used for checkpoint selection): 91.9980%
Test Accuracy (true held-out estimate):        92.0161%
Optimism Bias (val - test):                    -0.0181%
✓  Val and test acc are close — data split looks healthy.


In [13]:
# ─── SAVE RESULTS ────────────────────────────────────────────────────────────
results_path = "/kaggle/working/test1_results.txt"
with open(results_path, "w") as f:
    f.write("Test 1: Proper 3-Way Split Results\n")
    f.write("="*50 + "\n")
    f.write(f"Split:         {args.train_ratio*100:.0f}% train / "
            f"{args.val_ratio*100:.0f}% val / "
            f"{(1-args.train_ratio-args.val_ratio)*100:.0f}% test\n")
    f.write(f"Test samples:  {len(test_ds):,}\n")
    f.write(f"Test Accuracy: {test_acc:.4%}\n")
    f.write(f"Val  Accuracy: {val_acc:.4%}\n")
    f.write(f"Optimism Bias: {val_acc - test_acc:+.4%}\n\n")
    f.write(classification_report(
        test_labels, test_preds,
        target_names=['Safe (0)', 'Malicious (1)'],
        digits=4
    ))
    f.write(f"\nConfusion Matrix:\n")
    f.write(f"  TN={tn}  FP={fp}\n  FN={fn}  TP={tp}\n")

print(f"Results saved to {results_path}")

Results saved to /kaggle/working/test1_results.txt
